# TP3 — Lab 3: Text Normalization

## Exercise 1 — Lowercasing and its limits

In [1]:
import re

text = "NLP Is Used In Many Applications."
print(text.lower())

text2 = "Apple released a new iPhone; the apple I ate was sweet."
print(text2.lower())

print("\n3) In Named Entity Recognition Capitalization is one of the strongest clues, US vs us, Turkey vs turkey, and Bush vs bush")

def smart_lower(text):
    def fix(match: re.Match):
        w = match.group()
        if w.isupper() and 2 <= len(w) <= 5:
            return w
        return w.lower()
    return re.sub(r"\w+", fix, text)

print(smart_lower("\n4) The NLP course covers USA-based case studies."))
print(smart_lower("WE ARE FINE"))


nlp is used in many applications.
apple released a new iphone; the apple i ate was sweet.

3) In Named Entity Recognition Capitalization is one of the strongest clues, US vs us, Turkey vs turkey, and Bush vs bush

4) the NLP course covers USA-based case studies.
WE ARE FINE


### Answers — Exercise 1

1. **Q1 — Why is lowercasing useful?** It merges case variants of the same word (Apple/apple, NLP/nlp) into one feature. The vocabulary gets smaller and a search query matches regardless of case.
2. **Q2 — When is case important?** In NER: "Apple released a new iPhone" vs "I ate an apple". After lowercasing, the company and the fruit are identical, so the model loses a strong clue. Same for US/us, Turkey/turkey, Bush/bush.
3. **Q3 — Weakness of `smart_lower()`?** It keeps *any* 2–5 letter all-caps word, not only acronyms: "WE ARE FINE" stays uppercase. It also lowercases real acronyms longer than 5 letters (UNESCO) and turns the pronoun "I" into "i".

## Exercise 2 — Whitespace and invisible-character normalization

In [2]:
import re

text = "NLP   is \n\n very    useful."
result = re.sub(r"\s+", " ", text).strip()
print(result)

messy = "NLP\tis\n\nvery\u00a0\u00a0useful."
print(re.sub(r"\s+", " ", messy).strip())

def normalize_whitespace(text):
    text = text.replace("\u00a0", " ")

    lines = [line.strip() for line in text.splitlines()]
    cleaned = " ".join(line for line in lines if line)

    return re.sub(r"\s+", " ", cleaned).strip()

para = """   Natural language   processing
        is a field of AI \t
   that studies\u00a0human language.   """

print(normalize_whitespace(para))


NLP is very useful.
NLP is very useful.
Natural language processing is a field of AI that studies human language.


### Answers — Exercise 2

1. **Q1 — What is fixed, and what is `\s+`?** Irregular spacing (spaces, tabs, newlines, non-breaking spaces) is collapsed to a single space. `\s+` matches one or more whitespace characters; in Python 3 this includes Unicode spaces such as `\u00a0`.
2. **Q2 — Why do these appear in real text?** PDF/HTML extraction keeps the original layout, HTML uses `&nbsp;`, and line wrapping adds newlines and tabs.
3. **Q3 — Can this destroy meaning?** Yes: indentation is syntax in Python, line breaks matter in poetry, and collapsing spaces destroys table alignment.

## Exercise 3 — Number normalization: when and how

In [3]:
import re

text = "The course lasts 20 hours and contains 12 exercises."
print(re.sub(r"\d+", "<NUM>", text))

invoice = "The invoice #4521 totals $1,250.00, due in 30 days."
print(re.sub(r"\d+", "<NUM>", invoice))

pattern = r"\d+(?:,\d{3})*(?:\.\d+)?"
print(re.sub(pattern, "<NUM>", invoice))

def normalize_num(text):
    return re.sub(pattern, "<NUM>", text)

def normalize_num_magnitude(text):
    def repl(m):
        value = float(m.group().replace(",", ""))
        return "<NUM_SMALL>" if value < 100 else "<NUM_LARGE>"
    return re.sub(pattern, repl, text)

print(normalize_num(invoice))
print(normalize_num_magnitude(invoice))


The course lasts <NUM> hours and contains <NUM> exercises.
The invoice #<NUM> totals $<NUM>,<NUM>.<NUM>, due in <NUM> days.
The invoice #<NUM> totals $<NUM>, due in <NUM> days.
The invoice #<NUM> totals $<NUM>, due in <NUM> days.
The invoice #<NUM_LARGE> totals $<NUM_LARGE>, due in <NUM_SMALL> days.


### Answers — Exercise 3

- With plain `\d+`, `$1,250.00` becomes `$<NUM>,<NUM>.<NUM>`: the amount is split. The comma-aware pattern keeps it as one number.
- `<NUM>` is useful for topic classification or bag-of-words: it shrinks the vocabulary and avoids overfitting to rare numbers.
- Exact numbers should be kept in financial, medical or legal text, where the value itself is the information.
- For the invoice, `<NUM_SMALL>`/`<NUM_LARGE>` keeps more than `<NUM>` (we still know 1,250 is large and 30 is small), but exact comparison needs the real value. Best: extract the number and keep it.

## Exercise 4 — Building a full text-cleaning pipeline

In [4]:
import re
from nltk.tokenize import word_tokenize

# Patterns from Lab 1
email_pattern = r"[\w.+-]+@[A-Za-z0-9-]+(?:\.[A-Za-z0-9-]+)*\.[A-Za-z]{2,}"
url_pattern   = r"https?://[^\s,()]*[^\s,().]"

comment = """OMG this courseeee is SOOOO good!!! 10/10 would recommend :) Visit
https://univ.dz/nlp for more info. Contact: prof@univ.dz"""

def clean_text(text):
    text = text.lower()
    text = re.sub(r"\s+", " ", text).strip()

    text = re.sub(url_pattern, " ", text)
    text = re.sub(email_pattern, " ", text)

    text = re.sub(r"(.)\1{2,}", r"\1\1", text)
    return text

print("\ncomment:", comment)
print("\nclean_text(comment):", clean_text(comment))

orig_tokens = word_tokenize(comment)
clean_tokens = word_tokenize(clean_text(comment))

print("\ncomment tokens:", orig_tokens)
print("count:", len(orig_tokens), "| distinct:", len(set(orig_tokens)))

print("\ncleaned comment tokens:", clean_tokens)
print("count:", len(clean_tokens), "| distinct:", len(set(clean_tokens)))


comment: OMG this courseeee is SOOOO good!!! 10/10 would recommend :) Visit
https://univ.dz/nlp for more info. Contact: prof@univ.dz

clean_text(comment): omg this coursee is soo good!! 10/10 would recommend :) visit  for more info. contact:  

comment tokens: ['OMG', 'this', 'courseeee', 'is', 'SOOOO', 'good', '!', '!', '!', '10/10', 'would', 'recommend', ':', ')', 'Visit', 'https', ':', '//univ.dz/nlp', 'for', 'more', 'info', '.', 'Contact', ':', 'prof', '@', 'univ.dz']
count: 27 | distinct: 23

cleaned comment tokens: ['omg', 'this', 'coursee', 'is', 'soo', 'good', '!', '!', '10/10', 'would', 'recommend', ':', ')', 'visit', 'for', 'more', 'info', '.', 'contact', ':']
count: 20 | distinct: 18


### Answers — Exercise 4

- **Task 3 — vocabulary size:** before cleaning, 27 tokens (23 distinct); after cleaning, 20 tokens (18 distinct). Removing the URL/email and merging case variants shrinks the vocabulary.
- **Task 4 — should "!" and repeated letters be removed for sentiment?** Not completely. "!!!" and "soooo good" carry intensity, so removing them loses sentiment signal. Collapsing to two letters keeps some of it, but keeping a feature like the number of "!" or an `<EXCLAIM>` token is better. Removing them is acceptable for topic classification, not for sentiment.

## Exercise 5 — Normalization decisions across applications

| Text element | Application | Decision |
|---|---|---|
| Capital letters | Sentiment analysis | Normalize (lowercase), but keep an ALL-CAPS feature if emphasis matters |
| Numbers | Financial document analysis | Preserve |
| Punctuation | Question answering | Preserve |
| Multiple spaces / line breaks | Web-scraped corpus cleaning | Normalize |
| Emojis | Social-media sentiment analysis | Preserve |
| Repeated letters ("soooo good") | Social-media sentiment analysis | Normalize (collapse to at most 2) |
| Accented characters (naïve, café) | Multilingual search engine | Normalize (NFC) / fold accents for matching, keep original for display |

**Justifications**

- **Capital letters — sentiment:** lowercasing works for bag-of-words, but ALL CAPS can signal shouting, so a caps feature helps. The opposite choice (preserve case) is right for NER.
- **Numbers — finance:** exact amounts matter ($1,250.00 vs $1,205.00). The opposite choice (replace with `<NUM>`) is fine for topic classification.
- **Emojis — social sentiment:** 🙂 vs 😞 is direct sentiment signal. The opposite choice (remove them) is fine when cleaning formal text.
- **Accents — search:** folding "café" to "cafe" lets users type without accents and improves recall. The opposite choice (preserve) is right when accents distinguish words, as in "péché" vs "pêche".